# Notebook 03: Baseline Classifier — Main19

This notebook trains a reproducible EfficientNet-B0 baseline for the 19-class
multi-crop disease-classification task.

Data-use protocol:

- Training: `main19_train.csv` only
- Model selection and early stopping: `main19_calibration.csv` only
- Internal test: not loaded in this notebook
- Class weights: calculated from training data only

This run is provisional until the complete near-duplicate manual audit from
Notebook 02 is finalised.

In [ ]:
import os
import gc
import json
import random
import platform
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image, ImageFile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report
)

ImageFile.LOAD_TRUNCATED_IMAGES = True

SEED = 7

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("Torchvision:", __import__("torchvision").__version__)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")

OUTPUT_DIR = WORK_DIR / "baseline_main19_outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
METADATA_DIR = OUTPUT_DIR / "metadata"

for folder in [
    OUTPUT_DIR,
    CHECKPOINT_DIR,
    FIG_DIR,
    TABLE_DIR,
    METADATA_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Input root:", INPUT_ROOT)
print("Output directory:", OUTPUT_DIR)

In [ ]:
def find_input_file(file_name):
    matches = list(INPUT_ROOT.rglob(file_name))

    if len(matches) == 0:
        raise FileNotFoundError(
            f"Could not find '{file_name}' under /kaggle/input.\n"
            "Attach the Notebook-02 split_outputs dataset."
        )

    if len(matches) > 1:
        print(f"Multiple matches found for {file_name}:")
        for match in matches:
            print(" -", match)
        print("\nUsing:", matches[0])

    return matches[0]


TRAIN_MANIFEST_PATH = find_input_file("main19_train.csv")
CALIBRATION_MANIFEST_PATH = find_input_file("main19_calibration.csv")
CLASS_MAPPING_PATH = find_input_file("class_to_index_main19.json")
PROTOCOL_PATH = find_input_file("data_protocol.json")

print("Train manifest:", TRAIN_MANIFEST_PATH)
print("Calibration manifest:", CALIBRATION_MANIFEST_PATH)
print("Class mapping:", CLASS_MAPPING_PATH)
print("Data protocol:", PROTOCOL_PATH)